# MarkdownHeaderTextSplitter

마크다운 문서의 구조를 고려해 임베딩하면, 문서의 전반적인 맥락과 주제를 더 잘 담은 벡터 표현을 만들 수 있습니다.

그래서 마크다운 파일을 **헤더 단위**로 나누고, 각 청크에 **어떤 헤더 아래의 내용인지**를 메타데이터로 남기고 싶을 때가 많습니다.
`MarkdownHeaderTextSplitter` 는 지정한 헤더 집합을 기준으로 문서를 분할하고, 헤더 계층을 메타데이터로 기록합니다.

> **🔄 최신 버전 기준 변경 사항 (langchain-text-splitters 1.x)**
> - 핵심 API는 책과 동일합니다. (`headers_to_split_on`, `strip_headers`, `split_text`)
> - 줄 단위로 반환하는 `return_each_line` 옵션 예제를 추가했습니다.
> - 2단계 분할 시 `split_documents()` 를 써야 **헤더 메타데이터와 overlap이 올바르게 적용된다**는 공식 문서의 주의 사항을 추가했습니다.
> - 원본 서식(공백·줄바꿈·코드 블록)을 그대로 보존해야 하는 경우를 위한 `ExperimentalMarkdownSyntaxTextSplitter` 를 소개합니다.

In [ ]:
%pip install -qU langchain-text-splitters

- `markdown_document`: 마크다운 문자열
- `headers_to_split_on`: `(헤더 기호, 메타데이터 키 이름)` 튜플 리스트
- `split_text()`: 헤더 기준으로 분할해 `Document` 리스트를 반환

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document = "# Title\n\n## 1. SubTitle\n\nHi this is Jim\n\nHi this is Joe\n\n### 1-1. Sub-SubTitle \n\nHi this is Lance \n\n## 2. Baz\n\nHi this is Molly"
print(markdown_document)

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),    # 레벨 1 헤더 → metadata["Header 1"]
    ("##", "Header 2"),   # 레벨 2 헤더 → metadata["Header 2"]
    ("###", "Header 3"),  # 레벨 3 헤더 → metadata["Header 3"]
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)

for header in md_header_splits:
    print(f"{header.page_content}")
    print(f"{header.metadata}", end="\n=====================\n")

기본적으로 분할 기준이 된 헤더는 청크 본문에서 **제거**됩니다. `strip_headers=False` 로 본문에 남길 수 있습니다.

In [ ]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  # 헤더를 본문에 유지
)
md_header_splits = markdown_splitter.split_text(markdown_document)

for header in md_header_splits:
    print(f"{header.page_content}")
    print(f"{header.metadata}", end="\n=====================\n")

## 🔄 추가: 줄 단위로 반환하기 (`return_each_line`)

기본 동작은 같은 헤더 아래의 줄들을 하나의 청크로 **합칩니다**. `return_each_line=True` 이면 각 줄이 별도의 `Document` 가 되며, 헤더 정보는 메타데이터에 그대로 남습니다.

In [ ]:
line_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    return_each_line=True,
)
for doc in line_splitter.split_text(markdown_document):
    print(doc.page_content, "|", doc.metadata)

## 헤더 분할 후 크기 제한 (2단계 분할)

각 헤더 그룹 안에서 원하는 텍스트 분할기를 다시 적용할 수 있습니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

markdown_document = "# Intro \n\n## History \n\nMarkdown[9] is a lightweight markup language for creating formatted text using a plain-text editor. John Gruber created Markdown in 2004 as a markup language that is appealing to human readers in its source code form.[9] \n\nMarkdown is widely used in blogging, instant messaging, online forums, collaborative software, documentation pages, and readme files. \n\n## Rise and divergence \n\nAs Markdown popularity grew rapidly, many Markdown implementations appeared, driven mostly by the need for \n\nadditional features such as tables, footnotes, definition lists,[note 1] and Markdown inside HTML blocks. \n\n#### Standardization \n\nFrom 2012, a group of people, including Jeff Atwood and John MacFarlane, launched what Atwood characterised as a standardisation effort. \n\n# Implementations \n\nImplementations of Markdown are available for over a dozen programming languages."
print(markdown_document)

먼저 `MarkdownHeaderTextSplitter` 로 헤더 기준 분할을 합니다.

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    # ("##", "Header 2"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
md_header_splits = markdown_splitter.split_text(markdown_document)

for header in md_header_splits:
    print(f"{header.page_content}")
    print(f"{header.metadata}", end="\n=====================\n")

그 결과를 `RecursiveCharacterTextSplitter` 로 다시 분할합니다.

🔄 **공식 문서 주의 사항**
- 반드시 `split_text()` 가 아니라 **`split_documents(docs)`** 를 사용하세요. 그래야 섹션별 헤더 메타데이터가 각 청크에 유지됩니다.
- overlap은 **한 섹션이 `chunk_size` 를 넘어 여러 청크로 나뉠 때만** 적용되며, 섹션 경계를 넘어서 적용되지는 않습니다.

In [ ]:
chunk_size = 200
chunk_overlap = 20
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

for header in splits:
    print(f"{header.page_content}")
    print(f"{header.metadata}", end="\n=====================\n")

## 🔄 참고: 원본 서식 보존이 필요할 때

`MarkdownHeaderTextSplitter` 는 기본적으로 공백과 줄바꿈을 정리(strip)합니다.
코드 블록 들여쓰기나 원래 줄바꿈을 그대로 유지해야 한다면 같은 패키지의 `ExperimentalMarkdownSyntaxTextSplitter` 를 검토해 보세요.
(이름에 Experimental이 붙어 있지만 `langchain-text-splitters` 에 포함된 클래스이며, 지원 종료된 `langchain-experimental` 패키지와는 무관합니다.)

In [ ]:
from langchain_text_splitters import ExperimentalMarkdownSyntaxTextSplitter

syntax_splitter = ExperimentalMarkdownSyntaxTextSplitter()
for doc in syntax_splitter.split_text(markdown_document)[:3]:
    print(repr(doc.page_content))
    print(doc.metadata, end="\n=====================\n")